In [ ]:
GITHUB_REPO_URL = "https://github.com/24520152-ben/LLM-Generated-Multi-Level-Adversarial-Attacks-on-Vietnamese-Hate-Speech-Detection"
BASELINE_OPTIONS = {
    "model": "phobert",
    "epochs": 10,
    "device": "cuda:0",
}  # Use cuda:1 for the second GPU.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

kaggle_working = Path("/kaggle/working")
project_root = Path.cwd()
if kaggle_working.is_dir():
    project_root = kaggle_working / "ie403"
    if not project_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--", GITHUB_REPO_URL, str(project_root)],
            check=True,
        )
elif not (project_root / "src" / "vihsd_attack").is_dir():
    project_root = project_root.parent

source_dir = project_root / "src"
if not (source_dir / "vihsd_attack").is_dir():
    raise RuntimeError(f"Cannot find src/vihsd_attack in {project_root}.")
env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": str(project_root / ".venv"),
    "PYTHONPATH": str(source_dir),
}
if kaggle_working.is_dir() and not env.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.12"],
    cwd=project_root,
    env=env,
    check=True,
)

train_code = """
import json
import sys
from vihsd_attack.classifiers.service import run_baseline
from vihsd_attack.config import BaselineConfig

run_baseline(BaselineConfig(**json.loads(sys.argv[1])))
"""
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-u",
        "-c",
        train_code,
        json.dumps(BASELINE_OPTIONS, default=str),
    ],
    cwd=project_root,
    env=env,
    check=True,
)